In [16]:
import optuna
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [17]:
df = pd.read_csv('gurgaon_properties_feature_selection.csv')

In [18]:
df.rename(columns= {'servant room': 'servant_room', 'store room': 'store_room'}, inplace= True)

C:\Users\prama\AppData\Local\Temp\ipykernel_20644\1778438371.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns= {'servant room': 'servant_room', 'store room': 'store_room'}, inplace= True)


In [19]:
df['property_type'] = df['property_type'].str.capitalize()

In [20]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,facing,agePossession,built_up_area,servant_room,store_room,furnishing_type,luxury_category,floor_category
0,House,Sector 23,5.50,8,8,2,East,Moderately Old,2520.0,1,0,Semifurnished,Medium,Mid Floor
1,Flat,Sector 83,1.80,3,3,3,South,Relatively New,1553.0,1,0,Semifurnished,Medium,Mid Floor
2,Flat,Sector 83,1.30,3,3,1,North-West,Relatively New,2574.0,0,0,Semifurnished,Medium,Mid Floor
3,Flat,Sector 85,1.04,2,3,3+,West,Relatively New,1567.0,0,1,Unfurnished,Medium,Mid Floor
4,Flat,Sector 37d,0.80,2,2,3,South-East,Relatively New,1310.0,0,0,Unfurnished,High,Mid Floor


In [21]:
df['property_type'].value_counts()

property_type
Flat     11909
House     1225
Name: count, dtype: int64

In [22]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room']
encoding_cols = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category', 'facing']

In [23]:
feature_cols = numeric_cols + encoding_cols
X = df[feature_cols]
y = df['price']

In [24]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

## Ordinal Encoding

In [82]:
numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder())
])

In [83]:
preprocessor = ColumnTransformer(transformers=[
    ("Numeric",numeric_pipeline, numeric_cols ),
    ('Ordinal', ordinal_pipeline, encoding_cols)
])

In [84]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [85]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(lr_pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [86]:
scores.mean(),scores.std()

(np.float64(0.6880357956979648), np.float64(0.025699571804247614))

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [88]:
lr_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['bedRoom','bathroom','built_up_area',...,'luxury_category', 'floor_category','facing']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numeric', ...), ('Ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thr

In [89]:
y_pred = lr_pipeline.predict(X_test)

In [90]:
y_pred = np.expm1(y_pred)

In [91]:
mean_absolute_error(np.expm1(y_test),y_pred)

1.1152160226847692

In [92]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)

    if regressor_name == 'LinearRegression':
        model = LinearRegression()

    elif regressor_name == 'Ridge':
        model = Ridge()

    elif regressor_name == 'Lasso':
        model = Lasso()
    
    elif regressor_name == 'SVR':
        model = SVR()
        
    elif regressor_name == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()

    elif regressor_name == 'RandomForestRegressor':
        model = RandomForestRegressor()

    elif regressor_name == 'ExtraTreesRegressor':
        model = ExtraTreesRegressor()
    
    elif regressor_name == 'GradientBoostingRegressor':
        model = GradientBoostingRegressor()

    elif regressor_name == 'AdaBoostRegressor':
        model = AdaBoostRegressor()

    elif regressor_name == 'MLPRegressor':
        model = MLPRegressor()

    elif regressor_name == 'XGBRegressor':
        model = XGBRegressor()
        
    elif regressor_name == 'LGBMRegressor':
        model = LGBMRegressor()

    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [93]:
N_TRIALS = {
    'LinearRegression': 2, 'Ridge': 2, 'Lasso': 10, 'SVR': 5,
    'DecisionTreeRegressor': 2, 'RandomForestRegressor': 2, 'ExtraTreesRegressor': 2,
    'GradientBoostingRegressor': 2, 'AdaBoostRegressor': 2, 'MLPRegressor': 2,
    'XGBRegressor': 2, 'LGBMRegressor': 2,
}


all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# Filter only completed trials and rename columns to ['name', 'r2', 'mae']
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE'
})

Training LinearRegression (2 trials)...


Best trial: 0. Best value: 0.688036: 100%|██████████| 2/2 [00:04<00:00,  2.37s/it]


Training Ridge (2 trials)...


Best trial: 0. Best value: 0.688036: 100%|██████████| 2/2 [00:00<00:00,  6.40it/s]


Training Lasso (10 trials)...


Best trial: 0. Best value: 0.0542022: 100%|██████████| 10/10 [00:01<00:00,  6.65it/s]


... (17 outputs left out)

Best trial: 0. Best value: 0.891371:  50%|█████     | 1/2 [00:01<00:01,  1.73s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000464 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 421
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 13
[LightGBM] [Info] Start training from score 1.197784


Best trial: 0. Best value: 0.891371: 100%|██████████| 2/2 [00:03<00:00,  1.65s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000397 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 421
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 13
[LightGBM] [Info] Start training from score 1.197784


In [94]:
# Keep the best trial (lowest MAE) for each model name
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

model_df

,Name,R2,MAE
0,XGBRegressor,0.903373,0.519182
1,RandomForestRegressor,0.894105,0.547743
2,LGBMRegressor,0.891371,0.563530
3,ExtraTreesRegressor,0.871462,0.613279
4,GradientBoostingRegressor,0.856046,0.707393
5,DecisionTreeRegressor,0.790599,0.717220
6,MLPRegressor,0.760680,0.905023
7,SVR,0.739799,0.977711
8,AdaBoostRegressor,0.708255,1.023585
9,Ridge,0.688036,1.115147


## OneHotEncoding

In [95]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'sector','agePossession','furnishing_type', 'facing']
ohe_cols = ['sector','agePossession','furnishing_type', 'facing']

In [96]:
numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder())
])

ohe_pipeline = Pipeline(steps=[
    ('ohe', OneHotEncoder(drop= 'first'))
])

In [97]:
preprocessor = ColumnTransformer(transformers=[
    ("Numeric",numeric_pipeline, numeric_cols ),
    ("Ordinal", ordinal_pipeline, ordinal_cols),
    ("OHE", ohe_pipeline, ohe_cols)
])

In [98]:
# Creating a pipeline
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [99]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(lr_pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [100]:
scores.mean()

np.float64(0.8251820009553432)

In [101]:
scores.std()

np.float64(0.012618045654711694)

In [102]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [103]:
lr_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['bedRoom','bathroom','built_up_area',...,'luxury_category', 'floor_category','facing']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numeric', ...), ('Ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passe

In [104]:
y_pred = lr_pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
mean_absolute_error(np.expm1(y_test),y_pred)

0.7716859206910683

In [105]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)

    if regressor_name == 'LinearRegression':
        model = LinearRegression()

    elif regressor_name == 'Ridge':
        model = Ridge()

    elif regressor_name == 'Lasso':
        model = Lasso()
    
    elif regressor_name == 'SVR':
        model = SVR()
        
    elif regressor_name == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()

    elif regressor_name == 'RandomForestRegressor':
        model = RandomForestRegressor()

    elif regressor_name == 'ExtraTreesRegressor':
        model = ExtraTreesRegressor()
    
    elif regressor_name == 'GradientBoostingRegressor':
        model = GradientBoostingRegressor()

    elif regressor_name == 'AdaBoostRegressor':
        model = AdaBoostRegressor()

    elif regressor_name == 'MLPRegressor':
        model = MLPRegressor()

    elif regressor_name == 'XGBRegressor':
        model = XGBRegressor()
        
    elif regressor_name == 'LGBMRegressor':
        model = LGBMRegressor()

    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [106]:
N_TRIALS = {
    'LinearRegression': 2, 'Ridge': 2, 'Lasso': 10, 'SVR': 5,
    'DecisionTreeRegressor': 2, 'RandomForestRegressor': 2, 'ExtraTreesRegressor': 2,
    'GradientBoostingRegressor': 2, 'AdaBoostRegressor': 2, 'MLPRegressor': 2,
    'XGBRegressor': 2, 'LGBMRegressor': 2,
}


all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# Filter only completed trials and rename columns to ['name', 'r2', 'mae']
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE'
})



Training LinearRegression (2 trials)...


Best trial: 0. Best value: 0.825183: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]


Training Ridge (2 trials)...


Best trial: 0. Best value: 0.824957: 100%|██████████| 2/2 [00:00<00:00,  3.60it/s]


Training Lasso (10 trials)...


Best trial: 0. Best value: 0.0542022: 100%|██████████| 10/10 [00:02<00:00,  4.85it/s]


... (17 outputs left out)

Best trial: 0. Best value: 0.898533:  50%|█████     | 1/2 [00:01<00:01,  1.31s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000455 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 617
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 111
[LightGBM] [Info] Start training from score 1.197784


Best trial: 0. Best value: 0.898533: 100%|██████████| 2/2 [00:02<00:00,  1.30s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000413 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 617
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 111
[LightGBM] [Info] Start training from score 1.197784


In [107]:
# Keep the best trial (lowest MAE) for each model name
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

model_df

,Name,R2,MAE
0,ExtraTreesRegressor,0.909822,0.484060
1,RandomForestRegressor,0.905806,0.502818
2,XGBRegressor,0.902150,0.517024
3,LGBMRegressor,0.898533,0.545112
4,MLPRegressor,0.866225,0.605379
5,DecisionTreeRegressor,0.819808,0.649495
6,GradientBoostingRegressor,0.859128,0.675460
7,LinearRegression,0.825183,0.771686
8,Ridge,0.824957,0.774646
9,SVR,0.762611,0.932304


## One Hot Encoding With PCA

In [108]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'sector','agePossession','furnishing_type', 'facing']
ohe_cols = ['sector','agePossession']

In [109]:
numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder())
])

ohe_pipeline = Pipeline(steps=[
    ('ohe', OneHotEncoder(drop= 'first', sparse_output= False))
])

In [110]:
preprocessor = ColumnTransformer(transformers=[
    ("Numeric",numeric_pipeline, numeric_cols ),
    ("Ordinal", ordinal_pipeline, ordinal_cols),
    ("OHE", ohe_pipeline, ohe_cols)
])

In [111]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [112]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(lr_pipeline, X, y_transformed, cv=kfold, scoring='r2')


In [113]:
scores.mean()

np.float64(0.05817590506406463)

In [114]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)

    if regressor_name == 'LinearRegression':
        model = LinearRegression()

    elif regressor_name == 'Ridge':
        model = Ridge()

    elif regressor_name == 'Lasso':
        model = Lasso()
    
    elif regressor_name == 'SVR':
        model = SVR()
        
    elif regressor_name == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()

    elif regressor_name == 'RandomForestRegressor':
        model = RandomForestRegressor()

    elif regressor_name == 'ExtraTreesRegressor':
        model = ExtraTreesRegressor()
    
    elif regressor_name == 'GradientBoostingRegressor':
        model = GradientBoostingRegressor()

    elif regressor_name == 'AdaBoostRegressor':
        model = AdaBoostRegressor()

    elif regressor_name == 'MLPRegressor':
        model = MLPRegressor()

    elif regressor_name == 'XGBRegressor':
        model = XGBRegressor()
        
    elif regressor_name == 'LGBMRegressor':
        model = LGBMRegressor()

    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [115]:
N_TRIALS = {
    'LinearRegression': 2, 'Ridge': 2, 'Lasso': 10, 'SVR': 5,
    'DecisionTreeRegressor': 2, 'RandomForestRegressor': 2, 'ExtraTreesRegressor': 2,
    'GradientBoostingRegressor': 2, 'AdaBoostRegressor': 2, 'MLPRegressor': 2,
    'XGBRegressor': 2, 'LGBMRegressor': 2,
}
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# Filter only completed trials and rename columns to ['name', 'r2', 'mae']
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE'
})



Training LinearRegression (2 trials)...


Best trial: 0. Best value: 0.825183: 100%|██████████| 2/2 [00:01<00:00,  1.04it/s]


Training Ridge (2 trials)...


Best trial: 0. Best value: 0.825173: 100%|██████████| 2/2 [00:00<00:00,  4.54it/s]


Training Lasso (10 trials)...


Best trial: 0. Best value: 0.0542022: 100%|██████████| 10/10 [00:02<00:00,  4.52it/s]


... (17 outputs left out)

Best trial: 0. Best value: 0.898618:  50%|█████     | 1/2 [00:01<00:01,  1.17s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 599
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 102
[LightGBM] [Info] Start training from score 1.197784


Best trial: 0. Best value: 0.898618: 100%|██████████| 2/2 [00:02<00:00,  1.18s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000674 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 599
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 102
[LightGBM] [Info] Start training from score 1.197784


In [116]:
# Keep the best trial (lowest MAE) for each model name
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

model_df

,Name,R2,MAE
0,ExtraTreesRegressor,0.910964,0.478878
1,RandomForestRegressor,0.906228,0.489975
2,XGBRegressor,0.902761,0.522718
3,LGBMRegressor,0.898618,0.543148
4,MLPRegressor,0.863274,0.603876
5,DecisionTreeRegressor,0.819458,0.662195
6,GradientBoostingRegressor,0.858739,0.677739
7,LinearRegression,0.825183,0.770032
8,Ridge,0.825173,0.773105
9,SVR,0.762209,0.932369


## Target Encoder

In [117]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'sector','agePossession','furnishing_type', ]
ohe_cols = ['agePossession', 'facing']
target_enc_cols = ['sector']

In [118]:
numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder())
])

ohe_pipeline = Pipeline(steps=[
    ('ohe', OneHotEncoder(drop= 'first', sparse_output= False))
])
target_enc_pipeline = Pipeline(steps=[
    ('target_enc', TargetEncoder())
])

In [119]:
preprocessor = ColumnTransformer(transformers=[
    ("Numeric",numeric_pipeline, numeric_cols ),
    ("Ordinal", ordinal_pipeline, ordinal_cols),
    ("OHE", ohe_pipeline, ohe_cols),
    ("Target", target_enc_pipeline, target_enc_cols)
])

In [120]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [121]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(lr_pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [122]:
scores.mean(), scores.std()

(np.float64(0.8079210322140943), np.float64(0.015069248658413078))

In [123]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)

    if regressor_name == 'LinearRegression':
        model = LinearRegression()

    elif regressor_name == 'Ridge':
        model = Ridge()

    elif regressor_name == 'Lasso':
        model = Lasso()
    
    elif regressor_name == 'SVR':
        model = SVR()
        
    elif regressor_name == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()

    elif regressor_name == 'RandomForestRegressor':
        model = RandomForestRegressor()

    elif regressor_name == 'ExtraTreesRegressor':
        model = ExtraTreesRegressor()
    
    elif regressor_name == 'GradientBoostingRegressor':
        model = GradientBoostingRegressor()

    elif regressor_name == 'AdaBoostRegressor':
        model = AdaBoostRegressor()

    elif regressor_name == 'MLPRegressor':
        model = MLPRegressor()

    elif regressor_name == 'XGBRegressor':
        model = XGBRegressor()
        
    elif regressor_name == 'LGBMRegressor':
        model = LGBMRegressor()

    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [124]:
N_TRIALS = {
    'LinearRegression': 2, 'Ridge': 2, 'Lasso': 10, 'SVR': 5,
    'DecisionTreeRegressor': 2, 'RandomForestRegressor': 2, 'ExtraTreesRegressor': 2,
    'GradientBoostingRegressor': 2, 'AdaBoostRegressor': 2, 'MLPRegressor': 2,
    'XGBRegressor': 2, 'LGBMRegressor': 2,
}
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# Filter only completed trials and rename columns to ['name', 'r2', 'mae']
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE'
})



Training LinearRegression (2 trials)...


Best trial: 0. Best value: 0.807921: 100%|██████████| 2/2 [00:00<00:00,  4.61it/s]


Training Ridge (2 trials)...


Best trial: 0. Best value: 0.807924: 100%|██████████| 2/2 [00:00<00:00,  5.30it/s]


Training Lasso (10 trials)...


Best trial: 0. Best value: 0.0542022: 100%|██████████| 10/10 [00:01<00:00,  5.56it/s]


... (17 outputs left out)

Best trial: 0. Best value: 0.902094:  50%|█████     | 1/2 [00:01<00:01,  1.23s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000366 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 544
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 24
[LightGBM] [Info] Start training from score 1.197784


Best trial: 0. Best value: 0.902094: 100%|██████████| 2/2 [00:02<00:00,  1.24s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000396 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 544
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 24
[LightGBM] [Info] Start training from score 1.197784


In [125]:
# Keep the best trial (lowest MAE) for each model name
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

model_df

,Name,R2,MAE
0,RandomForestRegressor,0.911840,0.474540
1,ExtraTreesRegressor,0.911931,0.483493
2,XGBRegressor,0.907405,0.509694
3,LGBMRegressor,0.902094,0.526528
4,GradientBoostingRegressor,0.878420,0.617432
5,DecisionTreeRegressor,0.828299,0.620740
6,MLPRegressor,0.807544,0.700970
7,LinearRegression,0.807921,0.824591
8,Ridge,0.807924,0.824670
9,SVR,0.792047,0.868703


## Mix Encoding - 1

In [131]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'furnishing_type', 'agePossession']
ohe_cols = ['sector', 'facing', 'agePossession']
target_enc_cols = ['sector']

In [132]:
ordinal_categories = [
    ['Flat', 'House'],                                                                 # property_type
    ['0', '1', '2', '3', '3+'],                                                        # balcony
    ['Low', 'Medium', 'High'],                                                         # luxury_category
    ['Low Floor', 'Mid Floor', 'High Floor'],                                          # floor_category
    ['Unfurnished', 'Semifurnished', 'Furnished'],                                     # furnishing_type
    ['Old Property', 'Moderately Old', 'Relatively New', 'New Property', 'Under Construction'] # agePossession
]

In [133]:
preprocessor = ColumnTransformer(
    transformers=[
        ('Numeric', StandardScaler(), numeric_cols),
        ('Ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('OHE', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), ohe_cols),
        ('Target', TargetEncoder(smoothing=10), target_enc_cols)
    ],
    remainder='drop'
)

In [134]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)

    if regressor_name == 'LinearRegression':
        model = LinearRegression()

    elif regressor_name == 'Ridge':
        model = Ridge()

    elif regressor_name == 'Lasso':
        model = Lasso()
    
    elif regressor_name == 'SVR':
        model = SVR()
        
    elif regressor_name == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()

    elif regressor_name == 'RandomForestRegressor':
        model = RandomForestRegressor()

    elif regressor_name == 'ExtraTreesRegressor':
        model = ExtraTreesRegressor()
    
    elif regressor_name == 'GradientBoostingRegressor':
        model = GradientBoostingRegressor()

    elif regressor_name == 'AdaBoostRegressor':
        model = AdaBoostRegressor()

    elif regressor_name == 'MLPRegressor':
        model = MLPRegressor()

    elif regressor_name == 'XGBRegressor':
        model = XGBRegressor()
        
    elif regressor_name == 'LGBMRegressor':
        model = LGBMRegressor()

    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [135]:
N_TRIALS = {
    'LinearRegression': 2, 'Ridge': 2, 'Lasso': 10, 'SVR': 5,
    'DecisionTreeRegressor': 2, 'RandomForestRegressor': 2, 'ExtraTreesRegressor': 2,
    'GradientBoostingRegressor': 2, 'AdaBoostRegressor': 2, 'MLPRegressor': 2,
    'XGBRegressor': 2, 'LGBMRegressor': 2,
}
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# Filter only completed trials and rename columns to ['name', 'r2', 'mae']
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE'
})



Training LinearRegression (2 trials)...


Best trial: 0. Best value: 0.826407: 100%|██████████| 2/2 [00:05<00:00,  2.78s/it]


Training Ridge (2 trials)...


Best trial: 0. Best value: 0.826442: 100%|██████████| 2/2 [00:00<00:00,  3.73it/s]


Training Lasso (10 trials)...


Best trial: 0. Best value: -0.00117069: 100%|██████████| 10/10 [00:02<00:00,  3.78it/s]


... (17 outputs left out)

Best trial: 0. Best value: 0.90302:  50%|█████     | 1/2 [00:01<00:01,  1.58s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000499 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 610
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 110
[LightGBM] [Info] Start training from score 1.197784


Best trial: 0. Best value: 0.90302: 100%|██████████| 2/2 [00:03<00:00,  1.61s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000695 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 610
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 110
[LightGBM] [Info] Start training from score 1.197784


In [136]:
# Keep the best trial (lowest MAE) for each model name
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

model_df

,Name,R2,MAE
0,ExtraTreesRegressor,0.918904,0.462876
1,RandomForestRegressor,0.914107,0.467670
2,XGBRegressor,0.909023,0.496801
3,LGBMRegressor,0.903020,0.530041
4,SVR,0.896813,0.546819
5,MLPRegressor,0.886932,0.559132
6,DecisionTreeRegressor,0.838639,0.614719
7,GradientBoostingRegressor,0.877722,0.620448
8,LinearRegression,0.826407,0.770499
9,Ridge,0.826442,0.771302


## Mix Encoding - 2

In [146]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'furnishing_type']
ohe_cols = ['sector', 'facing', 'agePossession']
target_enc_cols = ['sector']

In [147]:
ordinal_categories = [
    ['Flat', 'House'],                                  # property_type
    ['0', '1', '2', '3', '3+'],                         # balcony
    ['Low', 'Medium', 'High'],                          # luxury_category
    ['Low Floor', 'Mid Floor', 'High Floor'],           # floor_category
    ['Unfurnished', 'Semifurnished', 'Furnished']       # furnishing_type
]

In [148]:
preprocessor = ColumnTransformer(
    transformers=[
        ('Numeric', StandardScaler(), numeric_cols),
        ('Ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('OHE', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), ohe_cols),
        ('Target', TargetEncoder(smoothing=10), target_enc_cols)
    ],
    remainder='drop'
)

In [149]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)

    if regressor_name == 'LinearRegression':
        model = LinearRegression()

    elif regressor_name == 'Ridge':
        model = Ridge()

    elif regressor_name == 'Lasso':
        model = Lasso()
    
    elif regressor_name == 'SVR':
        model = SVR()
        
    elif regressor_name == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()

    elif regressor_name == 'RandomForestRegressor':
        model = RandomForestRegressor()

    elif regressor_name == 'ExtraTreesRegressor':
        model = ExtraTreesRegressor()
    
    elif regressor_name == 'GradientBoostingRegressor':
        model = GradientBoostingRegressor()

    elif regressor_name == 'AdaBoostRegressor':
        model = AdaBoostRegressor()

    elif regressor_name == 'MLPRegressor':
        model = MLPRegressor()

    elif regressor_name == 'XGBRegressor':
        model = XGBRegressor()
        
    elif regressor_name == 'LGBMRegressor':
        model = LGBMRegressor()

    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [150]:
N_TRIALS = {
    'LinearRegression': 2, 'Ridge': 2, 'Lasso': 10, 'SVR': 5,
    'DecisionTreeRegressor': 2, 'RandomForestRegressor': 2, 'ExtraTreesRegressor': 2,
    'GradientBoostingRegressor': 2, 'AdaBoostRegressor': 2, 'MLPRegressor': 2,
    'XGBRegressor': 2, 'LGBMRegressor': 2,
}
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# Filter only completed trials and rename columns to ['name', 'r2', 'mae']
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE'
})



Training LinearRegression (2 trials)...


Best trial: 0. Best value: 0.826407: 100%|██████████| 2/2 [00:05<00:00,  2.88s/it]


Training Ridge (2 trials)...


Best trial: 0. Best value: 0.826442: 100%|██████████| 2/2 [00:00<00:00,  3.90it/s]


Training Lasso (10 trials)...


Best trial: 0. Best value: -0.00117069: 100%|██████████| 10/10 [00:02<00:00,  4.01it/s]


... (17 outputs left out)

Best trial: 0. Best value: 0.903215:  50%|█████     | 1/2 [00:01<00:01,  1.34s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000375 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 605
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 109
[LightGBM] [Info] Start training from score 1.197784


Best trial: 0. Best value: 0.903215: 100%|██████████| 2/2 [00:02<00:00,  1.37s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000414 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 605
[LightGBM] [Info] Number of data points in the train set: 10507, number of used features: 109
[LightGBM] [Info] Start training from score 1.197784


In [151]:
# Keep the best trial (lowest MAE) for each model name
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

model_df

,Name,R2,MAE
0,ExtraTreesRegressor,0.918259,0.465254
1,RandomForestRegressor,0.913322,0.468198
2,XGBRegressor,0.909455,0.502922
3,LGBMRegressor,0.903215,0.531979
4,SVR,0.898105,0.544585
5,MLPRegressor,0.884817,0.570373
6,DecisionTreeRegressor,0.843282,0.605944
7,GradientBoostingRegressor,0.877422,0.617010
8,LinearRegression,0.826407,0.770499
9,Ridge,0.826442,0.771299


## Selecting Top 3 Model & Hyperparameter Tuning
1. ExtraTreesRegressor
2. RandomForestRegressor
3. XGBRegressor

### Encoding

In [156]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'furnishing_type', 'agePossession']
ohe_cols = ['sector', 'facing', 'agePossession']
target_enc_cols = ['sector']

ordinal_categories = [
    ['Flat', 'House'],                                                                 # property_type
    ['0', '1', '2', '3', '3+'],                                                        # balcony
    ['Low', 'Medium', 'High'],                                                         # luxury_category
    ['Low Floor', 'Mid Floor', 'High Floor'],                                          # floor_category
    ['Unfurnished', 'Semifurnished', 'Furnished'],                                     # furnishing_type
    ['Old Property', 'Moderately Old', 'Relatively New', 'New Property', 'Under Construction'] # agePossession
]

preprocessor = ColumnTransformer(
    transformers=[
        ('Numeric', StandardScaler(), numeric_cols),
        ('Ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('OHE', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), ohe_cols),
        ('Target', TargetEncoder(smoothing=10), target_enc_cols)
    ],
    remainder='drop'
)

In [157]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
def objective(trial, regressor_name):
    # Store the model name so it appears in trials_dataframe()
    trial.set_user_attr("name", regressor_name)
    
    if regressor_name == 'RandomForestRegressor':
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])
        n_estimators = trial.suggest_int('n_estimators', 100, 800, step=50)
        max_depth = trial.suggest_int('max_depth', 5, 50)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        max_features = trial.suggest_float('max_features', 0.3, 1.0)
        max_samples = trial.suggest_float('max_samples', 0.5, 1.0) if bootstrap else None
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            max_samples=max_samples,
            n_jobs=-1, random_state=42
        )

    elif regressor_name == 'ExtraTreesRegressor':
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])
        n_estimators = trial.suggest_int('n_estimators', 100, 800, step=50)
        max_depth = trial.suggest_int('max_depth', 5, 50)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        max_features = trial.suggest_float('max_features', 0.3, 1.0)
        max_samples = trial.suggest_float('max_samples', 0.5, 1.0) if bootstrap else None
        # FIXED: Assigned to `model =` instead of `return`
        model = ExtraTreesRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            max_samples=max_samples,
            n_jobs=-1, random_state=42
        )
    
    elif regressor_name == 'XGBRegressor':
        n_estimators = trial.suggest_int('n_estimators', 200, 1500, step=50)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        min_child_weight = trial.suggest_int('min_child_weight', 1, 10)
        subsample = trial.suggest_float('subsample', 0.5, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)
        gamma = trial.suggest_float('gamma', 0.0, 5.0)
        reg_alpha = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True)
        reg_lambda = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
        model = XGBRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_child_weight=min_child_weight,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            tree_method='hist', n_jobs=-1, random_state=42
        )
        


    # 3. Attach the chosen model to your encoding pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # 4. Calculate CV R2 Score
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2', n_jobs=-1)
    r2_mean = scores.mean()

    # 5. Fit on train split & calculate MAE on test split
    pipeline.fit(X_train, y_train)
    y_pred = np.expm1(pipeline.predict(X_test))
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    # Store MAE inside the trial so we can put it in the DataFrame later
    trial.set_user_attr("mae", mae)

    return r2_mean

In [158]:
N_TRIALS = {
    'RandomForestRegressor': 35,
    'ExtraTreesRegressor': 30,
    'XGBRegressor': 100
}

all_trials = []

for model_name, n_trials in N_TRIALS.items():
    print(f"Training {model_name} ({n_trials} trials)...")
    study = optuna.create_study(direction="maximize")
    
    # Pass model_name via lambda so Optuna's TPESampler works at full strength
    study.optimize(lambda trial: objective(trial, model_name), n_trials=n_trials, show_progress_bar=True)
    all_trials.append(study.trials_dataframe())

Training RandomForestRegressor (35 trials)...


Best trial: 33. Best value: 0.921383: 100%|██████████| 35/35 [15:51<00:00, 27.17s/it]


Training ExtraTreesRegressor (30 trials)...


Best trial: 22. Best value: 0.920145: 100%|██████████| 30/30 [23:36<00:00, 47.21s/it] 


Training XGBRegressor (100 trials)...


Best trial: 94. Best value: 0.919629: 100%|██████████| 100/100 [12:02<00:00,  7.22s/it]


In [159]:
# Combine all trials into a single DataFrame
trials_df = pd.concat(all_trials, ignore_index=True)

# 1. Identify all parameter columns
param_cols = [col for col in trials_df.columns if col.startswith('params_')]

# 2. Pack non-NaN parameters into a single dictionary column
trials_df['best_params'] = trials_df[param_cols].apply(
    lambda row: {col.replace('params_', ''): val for col, val in row.dropna().items()},
    axis=1
)

# 3. Include 'best_params' in your selection
results_df = trials_df[trials_df['state'] == 'COMPLETE'][
    ['user_attrs_name', 'value', 'user_attrs_mae', 'best_params']
].rename(columns={
    'user_attrs_name': 'Name',
    'value': 'R2',
    'user_attrs_mae': 'MAE',
    'best_params': 'Best_Params'
})

# Note: Ensure column casing matches ('MAE' and 'Name', not 'mae' and 'name')
model_df = (
    results_df.sort_values('MAE', ascending=True)
    .drop_duplicates(subset=['Name'], keep='first')
    .reset_index(drop=True)
)

In [160]:
model_df

,Name,R2,MAE,Best_Params
0,RandomForestRegressor,0.917554,0.451601,"{'bootstrap': False, 'max_depth': 40, 'max_fea..."
1,ExtraTreesRegressor,0.919130,0.463349,"{'bootstrap': False, 'max_depth': 47, 'max_fea..."
2,XGBRegressor,0.919629,0.473232,"{'max_depth': 10, 'n_estimators': 1050, 'colsa..."


In [ ]:
model_df.iloc[0]['Best_Params']

{'bootstrap': False,
 'max_depth': 40,
 'max_features': 0.6282872618360571,
 'min_samples_leaf': 1.0,
 'min_samples_split': 4.0,
 'n_estimators': 400}

## Tuning the best model

In [25]:
numeric_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room']
ordinal_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category', 'furnishing_type', 'agePossession']
ohe_cols = ['sector', 'facing', 'agePossession']
target_enc_cols = ['sector']

ordinal_categories = [
    ['Flat', 'House'],                                                                 # property_type
    ['0', '1', '2', '3', '3+'],                                                        # balcony
    ['Low', 'Medium', 'High'],                                                         # luxury_category
    ['Low Floor', 'Mid Floor', 'High Floor'],                                          # floor_category
    ['Unfurnished', 'Semifurnished', 'Furnished'],                                     # furnishing_type
    ['Old Property', 'Moderately Old', 'Relatively New', 'New Property', 'Under Construction'] # agePossession
]

preprocessor = ColumnTransformer(
    transformers=[
        ('Numeric', StandardScaler(), numeric_cols),
        ('Ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('OHE', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), ohe_cols),
        ('Target', TargetEncoder(smoothing=10), target_enc_cols)
    ],
    remainder='drop'
)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)  # 5 folds is usually enough for tuning

def objective(trial):
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 800, step=50),
        max_depth=trial.suggest_int('max_depth', 5, 50),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        max_features=trial.suggest_float('max_features', 0.3, 1.0),
        bootstrap=bootstrap,
        max_samples=trial.suggest_float('max_samples', 0.5, 1.0) if bootstrap else None,
        n_jobs=-1, random_state=42,
    )
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(**params)),
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=kfold, scoring='r2')
    return scores.mean()

In [27]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100, show_progress_bar= True)

Best trial: 96. Best value: 0.919627: 100%|██████████| 100/100 [56:44<00:00, 34.05s/it]


In [28]:
print(f"Best CV R2: {study.best_value:.4f}")
print("Best params:", study.best_params)

Best CV R2: 0.9196
Best params: {'bootstrap': False, 'n_estimators': 750, 'max_depth': 31, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 0.38058019569051804}


In [29]:
# Final, one-time evaluation on the untouched test set
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)),
])
best_pipeline.fit(X_train, y_train)
pred = best_pipeline.predict(X_test)
print("Test R2 (log scale):", r2_score(y_test, pred))
print("Test MAE (price units):", mean_absolute_error(np.expm1(y_test), np.expm1(pred)))

Test R2 (log scale): 0.9123746678314448
Test MAE (price units): 0.4491151641812467


## Validating

In [30]:
data = [['House', 'Sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'Unfurnished', 'Low', 'Low Floor', 'North-East']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant_room', 'store_room',
       'furnishing_type', 'luxury_category', 'floor_category', 'facing']

In [31]:
# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant_room,store_room,furnishing_type,luxury_category,floor_category,facing
0,House,Sector 102,4,3,3+,New Property,2750,0,0,Unfurnished,Low,Low Floor,North-East


In [32]:
np.expm1(best_pipeline.predict(one_df))

array([3.61018651])

In [33]:
import joblib

joblib.dump(best_pipeline, 'house_price_model.pkl', compress= 3)

['house_price_model.pkl']

In [55]:
X['bedRoom'].value_counts()

bedRoom
3     6133
4     3584
2     2462
5      488
1      204
6       96
9       54
7       43
10      39
8       31
Name: count, dtype: int64

In [57]:
X['bathroom'].value_counts()

bathroom
3     4363
4     3890
2     2795
5     1211
6      406
1      239
7       97
9       58
8       37
10      33
11       3
12       2
Name: count, dtype: int64

In [ ]:
X['bathroom'].value_counts()

In [56]:
X.head()

,bedRoom,bathroom,built_up_area,servant_room,store_room,property_type,sector,balcony,agePossession,furnishing_type,luxury_category,floor_category,facing
0,8,8,2520.0,1,0,House,Sector 23,2,Moderately Old,Semifurnished,Medium,Mid Floor,East
1,3,3,1553.0,1,0,Flat,Sector 83,3,Relatively New,Semifurnished,Medium,Mid Floor,South
2,3,3,2574.0,0,0,Flat,Sector 83,1,Relatively New,Semifurnished,Medium,Mid Floor,North-West
3,2,3,1567.0,0,1,Flat,Sector 85,3+,Relatively New,Unfurnished,Medium,Mid Floor,West
4,2,2,1310.0,0,0,Flat,Sector 37d,3,Relatively New,Unfurnished,High,Mid Floor,South-East


In [36]:
import joblib
joblib.dump(X, 'df.pkl')

['df.pkl']